# Use Case 2: Handwritten Digit Recognition using a CNN

This notebook demonstrates convolution, feature maps, pooling and automatic feature extraction.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

CNN layers are imported from TensorFlow Keras.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)


## Step 2: Configure the dataset

The same digit images are used so that the dense network and CNN can be compared.

In [ ]:
DATASET_PATH = "../datasets/06_handwritten_digits"
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32


## Step 3: Load image tensors

Images are automatically represented as numeric tensors.

In [ ]:
DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 4: Display samples

This shows the different handwriting classes.

In [ ]:
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for index in range(min(9, len(images))):
    plt.subplot(3, 3, index + 1)
    plt.imshow(images[index].numpy().astype("uint8"))
    plt.title(class_names[int(labels[index])])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 5: Build the CNN

`Conv2D` learns filters; `MaxPooling2D` reduces feature-map dimensions.

In [ ]:
model = Sequential([
    Input(shape=(64, 64, 3)),
    Rescaling(1.0 / 255),

    Conv2D(32, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(64, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(128, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(10, activation="softmax"),
])

model.summary()


## Step 6: Compile and train

The CNN learns image features and class boundaries together.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
)


## Step 7: Evaluate

Validation metrics show how well the CNN handles unseen handwriting.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()
